# 01 Parsing the RG I XML File

This notebook parses the XML-encoded entries of *Repertorium Germanicum I*

Goals:

1. Load and inspect the XML file
2. Count the main XML elements
3. Extract the main texts
4. Export the extracted texts as a CSV file for further analysis

# 1 Setup

In [1]:
from pathlib import Path
from collections import Counter
import re
import xml.etree.ElementTree as ET

import pandas as pd
from IPython.display import display

In [2]:
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)

In [3]:
def find_project_root(start_path):
    """
    Search the current directory and its parents for data/raw/rg1.xml.
    """
    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        xml_candidate = candidate / "data" / "raw" / "rg1.xml"

        if xml_candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find data/raw/rg1.xml. "
        "Check that rg1.xml is stored in the project's data/raw directory."
    )


project_root = find_project_root(Path.cwd())

xml_path = project_root / "data" / "raw" / "rg1.xml"
output_dir = project_root / "data" / "derived"

print("Current working directory:", Path.cwd())
print("Project root:", project_root)
print("XML path:", xml_path)
print("XML exists:", xml_path.exists())
print("Output directory:", output_dir)

Current working directory: c:\Digital Philology\measuring-manuscripts-project-template
Project root: C:\Digital Philology\measuring-manuscripts-project-template
XML path: C:\Digital Philology\measuring-manuscripts-project-template\data\raw\rg1.xml
XML exists: True
Output directory: C:\Digital Philology\measuring-manuscripts-project-template\data\derived


## 1. Loading and inspecting the XML

The XML is parsed as a hierarchical tree. Before extracting the text, the
root element, document volume, element counts, and available tag types are
inspected.

In [4]:
tree = ET.parse(xml_path)
root = tree.getroot()

print("Root tag:", root.tag)
print("Root attributes:", root.attrib)
print("File size:", round(xml_path.stat().st_size / 1_000_000, 2), "MB")

Root tag: rg
Root attributes: {'vol': '1'}
File size: 2.4 MB


In [5]:
core_elements = [
    "lemma",
    "reg",
    "head",
    "sublemma",
    "date",
    "fund",
    "ka",
    "kz",
    "personenindex",
    "ortsindex"
]

core_element_counts = []

for tag in core_elements:
    count = len(root.findall(f".//{tag}"))

    core_element_counts.append({
        "element": tag,
        "count": count
    })

core_counts_df = pd.DataFrame(core_element_counts)

display(core_counts_df)

,element,count
0,lemma,3845
1,reg,3845
2,head,3845
3,sublemma,5589
4,date,1103
5,fund,5260
6,ka,732
7,kz,732
8,personenindex,3845
9,ortsindex,3845


In [6]:
tag_counts = Counter(
    element.tag
    for element in root.iter()
    if isinstance(element.tag, str)
)

tag_inventory_df = pd.DataFrame(
    tag_counts.items(),
    columns=["tag", "count"]
).sort_values(
    "count",
    ascending=False
).reset_index(drop=True)

print("Number of distinct XML tag names:", len(tag_inventory_df))

display(tag_inventory_df.head(30))

Number of distinct XML tag names: 282


,tag,count
0,sublemma,5589
1,fund,5260
2,head,3845
3,reg,3845
4,lemma,3845
5,personenindex,3845
6,ortsindex,3845
7,abk296,3561
8,abk312,2959
9,abk671,1804


## 2. Text extraction rules

Two versions of every `<head>` and `<sublemma>` are created:

### `full_text`

Contains all visible textual content, including archival references.

### `analysis_text`

Preserves the original abbreviated text but removes content marked with
`<fund>`, because archival shelfmarks are metadata rather than linguistic
content.

The abbreviation elements themselves are removed as XML markup, but their
surface forms are retained.

In [7]:
def normalize_extracted_text(text):
    """
    Normalize whitespace and remove extraction artefacts around punctuation.
    """
    text = " ".join(text.split())

    # Avoid doubled full stops created when a <fund> element is removed
    # after an abbreviation that already ends in a full stop.
    text = re.sub(r"\.\s+\.", ".", text)

    # Remove whitespace before punctuation.
    text = re.sub(r"\s+([,;:!?])", r"\1", text)
    text = re.sub(r"\s+\.", ".", text)

    return text.strip()

In [8]:
def extract_text(element, skip_tags=None):
    """
    Recursively extract plain text from an XML element.

    Parameters
    ----------
    element:
        XML element from which text is extracted.

    skip_tags:
        Set of XML tag names whose content should be removed.
        For example: {"fund"}.

    Notes
    -----
    - Text inside abbreviation elements is preserved.
    - XML tags themselves are not included.
    - Empty <ka/> and <kz/> elements contribute no text.
    """
    if skip_tags is None:
        skip_tags = set()
    else:
        skip_tags = set(skip_tags)

    parts = []

    def walk(node):
        if node.text:
            parts.append(node.text)

        for child in node:
            if child.tag not in skip_tags:
                walk(child)

            # The tail occurs after the child element and must be preserved,
            # even when the child itself is skipped.
            if child.tail:
                parts.append(child.tail)

    walk(element)

    return normalize_extracted_text("".join(parts))

def clean_attribute(value):
    """
    Strip whitespace from an XML attribute.
    Return None for missing or empty values.
    """
    if value is None:
        return None

    value = value.strip()

    return value if value else None

first_lemma = root.find("./lemma")
first_head = first_lemma.find("./reg/head")
first_sublemma = first_lemma.find("./reg/sublemma")

print("HEAD")
print(extract_text(first_head))

print("\nFULL SUBLEMMA")
print(extract_text(first_sublemma))

print("\nSUBLEMMA WITHOUT <fund>")
print(extract_text(
    first_sublemma,
    skip_tags={"fund"}
))

HEAD
Abardus Alamannus

FULL SUBLEMMA
qui portavit litteras d. pape ex parte marchionis Moravie et revertitur ad ipsum cum litteris d. pape I 366 108v.

SUBLEMMA WITHOUT <fund>
qui portavit litteras d. pape ex parte marchionis Moravie et revertitur ad ipsum cum litteris d. pape.


## 3 Extract Structured data

1. segments: one row per <head> or <sublemma>
2. abbreviations: one row per abbreviation occurrence;
3. dates: one row per `<date>`;
4. sources: one row per `<fund>`.

All tables are connected through `lemma_id` and `segment_id`.

In [9]:
segment_rows = []
abbreviation_rows = []
date_rows = []
source_rows = []

for lemma in root.findall("./lemma"):
    lemma_id = lemma.get("id")

    reg = lemma.find("./reg")

    if reg is None:
        continue

    segments = []

    # The head is assigned segment order 0.
    head = reg.find("./head")

    if head is not None:
        segments.append({
            "section": "head",
            "segment_order": 0,
            "element": head
        })

    # Sublemmata are assigned orders 1, 2, 3, ...
    for order, sublemma in enumerate(
        reg.findall("./sublemma"),
        start=1
    ):
        segments.append({
            "section": "sublemma",
            "segment_order": order,
            "element": sublemma
        })

    for segment in segments:
        section = segment["section"]
        segment_order = segment["segment_order"]
        element = segment["element"]

        # Use the XML id when available.
        # Otherwise construct a stable fallback id.
        segment_id = (
            element.get("id")
            or f"{lemma_id}-{segment_order}"
        )

        # Full textual content, including archival references.
        full_text = extract_text(element)

        # Linguistic/NER text, excluding archival references.
        analysis_text = extract_text(
            element,
            skip_tags={"fund"}
        )

        abbreviations = [
            node
            for node in element.iter()
            if isinstance(node.tag, str)
            and node.tag.startswith("abk")
        ]

        dates = list(element.iter("date"))
        sources = list(element.iter("fund"))

        ka_count = sum(
            1
            for node in element.iter()
            if node.tag == "ka"
        )

        kz_count = sum(
            1
            for node in element.iter()
            if node.tag == "kz"
        )

        # This is a simple whitespace-based count.
        # More detailed tokenization can be introduced in Notebook 02.
        word_count = len(analysis_text.split())

        segment_rows.append({
            "lemma_id": lemma_id,
            "segment_id": segment_id,
            "section": section,
            "segment_order": segment_order,
            "full_text": full_text,
            "analysis_text": analysis_text,
            "word_count": word_count,
            "character_count": len(analysis_text),
            "abbreviation_count": len(abbreviations),
            "abbreviation_density": (
                len(abbreviations) / word_count
                if word_count > 0
                else 0
            ),
            "date_count": len(dates),
            "source_count": len(sources),
            "ka_count": ka_count,
            "kz_count": kz_count
        })

        # One row per abbreviation occurrence.
        for position, abbreviation in enumerate(
            abbreviations,
            start=1
        ):
            abbreviation_rows.append({
                "lemma_id": lemma_id,
                "segment_id": segment_id,
                "section": section,
                "position": position,
                "tag": abbreviation.tag,
                "code": abbreviation.tag[3:],
                "surface": extract_text(abbreviation)
            })

        # One row per date occurrence.
        for position, date in enumerate(
            dates,
            start=1
        ):
            date_rows.append({
                "lemma_id": lemma_id,
                "segment_id": segment_id,
                "section": section,
                "position": position,
                "text": extract_text(date),
                "iso": clean_attribute(date.get("iso")),
                "year": clean_attribute(date.get("year"))
            })

        # One row per archival-source occurrence.
        for position, source in enumerate(
            sources,
            start=1
        ):
            source_rows.append({
                "lemma_id": lemma_id,
                "segment_id": segment_id,
                "section": section,
                "position": position,
                "text": extract_text(source),
                "iso": clean_attribute(source.get("iso")),
                "l1": clean_attribute(source.get("l1")),
                "l2": clean_attribute(source.get("l2")),
                "l3": clean_attribute(source.get("l3"))
            })



In [10]:
segments_df = pd.DataFrame(segment_rows)
abbreviations_df = pd.DataFrame(abbreviation_rows)
dates_df = pd.DataFrame(date_rows)
sources_df = pd.DataFrame(source_rows)

print("Segments:", len(segments_df))
print("Abbreviations:", len(abbreviations_df))
print("Dates:", len(dates_df))
print("Sources:", len(sources_df))

Segments: 9434
Abbreviations: 30118
Dates: 1103
Sources: 5260


## 4. Validation

The extracted tables are validated against expected counts from the original
XML. This checks whether records were lost or duplicated during parsing.

In [11]:
expected_counts = {
    "lemma": 3845,
    "head": 3845,
    "sublemma": 5589,
    "segment": 9434,
    "abbreviation": 30118,
    "date": 1103,
    "source": 5260
}

observed_counts = {
    "lemma": segments_df["lemma_id"].nunique(),
    "head": (
        segments_df["section"] == "head"
    ).sum(),
    "sublemma": (
        segments_df["section"] == "sublemma"
    ).sum(),
    "segment": len(segments_df),
    "abbreviation": len(abbreviations_df),
    "date": len(dates_df),
    "source": len(sources_df)
}

validation_df = pd.DataFrame({
    "expected": expected_counts,
    "observed": observed_counts
})

validation_df["match"] = (
    validation_df["expected"]
    == validation_df["observed"]
)

display(validation_df)

,expected,observed,match
lemma,3845,3845,True
head,3845,3845,True
sublemma,5589,5589,True
segment,9434,9434,True
abbreviation,30118,30118,True
date,1103,1103,True
source,5260,5260,True


## 6. Exporting the derived data

The original XML remains unchanged in `data/raw/`.

The parser exports four derived CSV files:

- `rg1_segments.csv`
- `rg1_abbreviations.csv`
- `rg1_dates.csv`
- `rg1_sources.csv`

In [12]:
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:", output_dir)

segments_df.to_csv(
    output_dir / "rg1_segments.csv",
    index=False,
    encoding="utf-8-sig"
)

abbreviations_df.to_csv(
    output_dir / "rg1_abbreviations.csv",
    index=False,
    encoding="utf-8-sig"
)

dates_df.to_csv(
    output_dir / "rg1_dates.csv",
    index=False,
    encoding="utf-8-sig"
)

sources_df.to_csv(
    output_dir / "rg1_sources.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV export completed.")

Output directory: C:\Digital Philology\measuring-manuscripts-project-template\data\derived
CSV export completed.


In [13]:
exported_files = []

for file_path in sorted(
    output_dir.glob("rg1_*.csv")
):
    exported_files.append({
        "file": file_path.name,
        "size_kb": round(
            file_path.stat().st_size / 1_000,
            2
        )
    })

exported_files_df = pd.DataFrame(exported_files)

display(exported_files_df)

,file,size_kb
0,rg1_abbreviations.csv,1457.64
1,rg1_dates.csv,65.07
2,rg1_segments.csv,1536.54
3,rg1_sources.csv,547.58


In [14]:
exported_segments_df = pd.read_csv(
    output_dir / "rg1_segments.csv"
)

print(
    "Rows read from exported CSV:",
    len(exported_segments_df)
)

display(exported_segments_df.head(3))

Rows read from exported CSV: 9434


,lemma_id,segment_id,section,segment_order,full_text,analysis_text,word_count,character_count,abbreviation_count,abbreviation_density,date_count,source_count,ka_count,kz_count
0,10100001,10100001-0,head,0,Abardus Alamannus,Abardus Alamannus,2,17,0,0.000000,0,0,0,0
1,10100001,10100001-1,sublemma,1,qui portavit litteras d. pape ex parte marchionis Moravie et revertitur ad ipsum cum litteris d. pape I 366 108v.,qui portavit litteras d. pape ex parte marchionis Moravie et revertitur ad ipsum cum litteris d. pape.,17,102,2,0.117647,0,1,0,0
2,10100002,10100002-0,head,0,Achatius Erhardi Asini de Monteferri cler. Salczburgen. dioc.,Achatius Erhardi Asini de Monteferri cler. Salczburgen. dioc.,8,61,2,0.250000,0,0,0,0
